In [1]:
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

In [4]:
df = pd.read_csv("/content/trainset-gpt5.1.csv")

In [11]:
!pip install -U pandas numpy torch transformers sentence-transformers


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 928.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26

In [5]:

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification


def _normalize_text(x: str) -> str:
    if x is None:
        return ""
    x = str(x).strip()
    # Collapse whitespace
    x = " ".join(x.split())
    return x


def _cosine_rowwise(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    # a,b are L2-normalized => cosine = dot
    return (a * b).sum(axis=1)


@torch.inference_mode()
def judge_same_answer(
    df: pd.DataFrame,
    answer_col: str = "answer",
    gpt_col: str = "gpt",
    question_col: str | None = None,   # set to your question column name if you have it
    context_col: str | None = None,    # set to your context column name if you have it
    emb_model_name: str = "BAAI/bge-m3",
    nli_model_name: str = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
    batch_size: int = 32,
    device: str | None = None,
    sim_threshold: float = 0.80,
    entail_threshold: float = 0.60,
) -> pd.DataFrame:
    """
    Returns df with:
      - sim_qaware: cosine similarity between embeddings of q+context+answer and q+context+gpt
      - ent_a_to_g: entailment prob for (qaware_answer -> qaware_gpt)
      - ent_g_to_a: entailment prob for (qaware_gpt -> qaware_answer)
      - same: True if both-direction entailment is high OR similarity is very high
      - confidence: combined score in [0,1] (rough, but useful)

    Why question/context: it makes short/elliptical answers comparable.
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    out = df.copy()

    # Build "q-aware" strings
    q = out[question_col].map(_normalize_text).tolist() if question_col else [""] * len(out)
    c = out[context_col].map(_normalize_text).tolist() if context_col else [""] * len(out)

    a = out[answer_col].map(_normalize_text).tolist()
    g = out[gpt_col].map(_normalize_text).tolist()

    def pack(qi, ci, ai):
        parts = []
        if qi:
            parts.append(f"Question: {qi}")
        if ci:
            parts.append(f"Context: {ci}")
        parts.append(f"Answer: {ai}")
        return "\n".join(parts)

    qa_a = [pack(qi, ci, ai) for qi, ci, ai in zip(q, c, a)]
    qa_g = [pack(qi, ci, gi) for qi, ci, gi in zip(q, c, g)]

    # ---------- Step 1: Embedding similarity ----------
    emb = SentenceTransformer(emb_model_name, device=device)

    emb_a = emb.encode(
        qa_a, batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
    )
    emb_g = emb.encode(
        qa_g, batch_size=batch_size, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
    )

    sim = _cosine_rowwise(emb_a, emb_g)
    out["sim_qaware"] = sim

    # ---------- Step 2: NLI entailment both directions ----------
    tok = AutoTokenizer.from_pretrained(nli_model_name)
    nli = AutoModelForSequenceClassification.from_pretrained(nli_model_name).to(device)
    nli.eval()

    # figure out label mapping robustly
    id2label = {int(k): v for k, v in nli.config.id2label.items()}
    # try common label variants
    def find_label_id(target: str):
        target = target.lower()
        for i, lab in id2label.items():
            if target in lab.lower():
                return i
        return None

    entail_id = find_label_id("entail")
    if entail_id is None:
        raise ValueError(f"Could not find entailment label in model labels: {id2label}")

    def nli_entail_probs(premises, hypotheses):
        probs_out = []
        for i in range(0, len(premises), batch_size):
            p = premises[i:i+batch_size]
            h = hypotheses[i:i+batch_size]
            enc = tok(p, h, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
            logits = nli(**enc).logits
            probs = torch.softmax(logits, dim=-1)[:, entail_id].detach().cpu().numpy()
            probs_out.append(probs)
        return np.concatenate(probs_out, axis=0)

    ent_a_to_g = nli_entail_probs(qa_a, qa_g)
    ent_g_to_a = nli_entail_probs(qa_g, qa_a)

    out["ent_a_to_g"] = ent_a_to_g
    out["ent_g_to_a"] = ent_g_to_a

    # Symmetric entailment score: minimum of both directions
    ent_sym = np.minimum(ent_a_to_g, ent_g_to_a)
    out["ent_sym"] = ent_sym

    # ---------- Decision rule ----------
    # If both entailments are high -> same
    # Or if similarity is extremely high -> same (covers paraphrases where NLI is shaky)
    same = (ent_sym >= entail_threshold) | (sim >= sim_threshold)
    out["same"] = same

    # Rough confidence: combine evidence (bounded to [0,1])
    # You can change weights. This is sane default.
    conf = np.clip(0.55 * ent_sym + 0.45 * sim, 0.0, 1.0)
    out["confidence"] = conf

    return out


# ---------------- Example ----------------
# df has columns: index, answer, gpt, and optionally question/context
# df_scored = judge_same_answer(df, answer_col="answer", gpt_col="gpt", question_col="question", context_col="context")
# df_scored[["answer","gpt","sim_qaware","ent_sym","same","confidence"]].head()


In [6]:
df

,answer,type,context,question,gpt
0,The U.S. Government has emphasized a policy of...,factual,While the housing and credit bubbles were buil...,What policy has U.S. Government emphasized fro...,Deregulation to encourage business.
1,The United States Air Force initiated their re...,factual,The United States had multiple rocket programs...,The US Air Force began research of ICBMs in wh...,1945
2,Beyoncé gave birth to her daughter on January ...,factual,"On January 7, 2012, Beyoncé gave birth to her ...",When did Beyoncé give birth to a daughter?,"January 7, 2012."
3,The New Forest is located on the opposite bank...,factual,The River Test runs along the western border o...,What forest is on the opposite bank of the Riv...,The New Forest.
4,Ninety-seven percent of male BYU graduates and...,factual,Some 97 percent of male BYU graduates and 32 p...,What percentage of graduates had taken a hiatu...,About 97% of male BYU graduates and 32% of fem...
...,...,...,...,...,...
10158,"The two publications, the 1998 Strategic Defen...",factual,NaN,Which part of the Britsh government were the t...,I don’t have the two publication titles or the...
10159,The French government believed that Jean-Claud...,factual,NaN,Who did the French government think was best c...,I’m missing the article or year you mean. Do y...
10160,Masjid Bilal is situated on Chimbarazoo Boulev...,factual,NaN,On what street is Masjid Bilal located?,There are many mosques named Masjid Bilal. Whi...
10161,The Taff Vale Case in 1901 boosted support for...,factual,NaN,What boosted support in 1901?,I don’t have the passage or topic—can you past...


In [ ]:
df_scored = judge_same_answer(df, answer_col="answer", gpt_col="gpt", question_col="question", context_col="context")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

In [ ]:
df_scored[["answer","gpt","sim_qaware","ent_sym","same","confidence"]].head()

In [ ]:
df_scored.to_csv("similarity_between_gpt_and_deberta.csv")